# BirdCLEF 2026 — Perch ONNX Inference

**Before running on Kaggle:**
1. Upload the files from `submission/time_dataset_kaggle/` as a new Kaggle dataset
2. Add that dataset + `rishikeshjani/perch-onnx-for-birdclef-2026` to this notebook
3. Update `MODEL_DATASET_SLUG` below to match your dataset slug

In [1]:
import subprocess
subprocess.run(['pip', 'install', '-q',
    '/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl'
], check=True)

CompletedProcess(args=['pip', 'install', '-q', '/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl'], returncode=0)

In [2]:
import os, json, time
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
import onnxruntime as ort
from pathlib import Path

MODEL_DATASET_SLUG = 'enrico777/v-3-4-agent-perch'

COMP_DIR       = Path('/kaggle/input/competitions/birdclef-2026')
ONNX_PATH      = Path('/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/perch_v2.onnx')
MODEL_DIR      = Path(f'/kaggle/input/datasets/{MODEL_DATASET_SLUG}')

SR           = 32_000
CLIP_SEC     = 5.0
CLIP_SAMPLES = int(SR * CLIP_SEC)

print(f'COMP_DIR exists: {COMP_DIR.exists()}')
print(f'ONNX exists:     {ONNX_PATH.exists()}')
print(f'MODEL_DIR exists:{MODEL_DIR.exists()}')

2026-05-17 17:38:51.406911: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779039531.698419      15 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779039531.780544      15 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779039532.457626      15 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779039532.457685      15 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779039532.457688      15 computation_placer.cc:177] computation placer alr

COMP_DIR exists: True
ONNX exists:     True
MODEL_DIR exists:True


In [3]:
# ── Load Perch ONNX ────────────────────────────────────────────────────────
so = ort.SessionOptions()
so.intra_op_num_threads = 4
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
sess     = ort.InferenceSession(str(ONNX_PATH), sess_options=so, providers=['CPUExecutionProvider'])
inp_name = sess.get_inputs()[0].name

dummy     = np.zeros((1, CLIP_SAMPLES), dtype=np.float32)
outs      = sess.run(None, {inp_name: dummy})
emb_idx   = next(i for i, o in enumerate(outs) if o.ndim == 2 and o.shape[-1] == 1536)
logit_idx = next(i for i, o in enumerate(outs) if o.ndim == 2 and o.shape[-1] > 5000)
print(f'Embedding: output[{emb_idx}] {outs[emb_idx].shape}')
print(f'Logits:    output[{logit_idx}] {outs[logit_idx].shape}')

Embedding: output[0] (1, 1536)
Logits:    output[3] (1, 14795)


In [4]:
# ── Load trained head (staged pipeline: 1d + pseudo-refine) ─────────────────
import json
from pathlib import Path
import numpy as np
import tensorflow as tf

model_dir = Path(MODEL_DIR)
print("Model files at:", model_dir)
for f in sorted(model_dir.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(model_dir)}  ({f.stat().st_size/1e6:.1f} MB)")

head_code_path = model_dir / "perch_best_head_code.py"
if not head_code_path.exists():
    head_code_path = model_dir / "best_head_code.py"

ns = {"tf": tf, "np": np}
exec(head_code_path.read_text(encoding="utf-8"), ns)
build_head = ns["build_head"]
get_training_config = ns["get_training_config"]

with open(model_dir / "species_cols.json", encoding="utf-8") as f:
    species_cols = json.load(f)

emb_dim = 1536
num_classes = len(species_cols)
head = build_head(emb_dim, num_classes)
print(f"Architecture built: {head.input_shape} → {head.output_shape}")

weights_candidates = [
    model_dir / "perch_final_head_pseudo.weights.h5",
    model_dir / "final_head_pseudo.weights.h5",
    model_dir / "perch_final_head.weights.h5",
    model_dir / "final_head.weights.h5",
    model_dir / "best_head.weights.h5",
]
weights_path = next((p for p in weights_candidates if p.exists()), None)

if weights_path is not None:
    head.load_weights(str(weights_path))
    print("Weights loaded OK from", weights_path.name)
else:
    keras_candidates = [
        model_dir / "perch_final_head_pseudo.keras",
        model_dir / "final_head_pseudo.keras",
        model_dir / "perch_final_head.keras",
        model_dir / "final_head.keras",
        model_dir / "best_head.keras",
    ]
    keras_path = next((p for p in keras_candidates if p.exists()), None)
    if keras_path is None:
        raise FileNotFoundError("No Perch head weights or .keras file in MODEL_DIR")
    head = tf.keras.models.load_model(str(keras_path), compile=False)
    print("Loaded full model from", keras_path.name)

cfg = get_training_config()
perch_weight = float(cfg.get("perch_weight", cfg.get("blend_weight", 0.2)))
print(f"perch_weight={perch_weight}  (from get_training_config)")

Model files at: /kaggle/input/datasets/enrico777/v-3-4-agent-perch
  bc_indices.npy  (0.0 MB)
  mapping_meta.json  (0.0 MB)
  perch_best_head_code.py  (0.0 MB)
  perch_final_head_pseudo.weights.h5  (26.8 MB)
  proxy_map.json  (0.0 MB)
  species_cols.json  (0.0 MB)
Architecture built: (None, 1536) → (None, 234)


2026-05-17 17:39:28.343803: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Weights loaded OK from perch_final_head_pseudo.weights.h5
perch_weight=0.3  (from get_training_config)


In [5]:
# ── Load species mapping ────────────────────────────────────────────────────
with open(model_dir / 'species_cols.json') as f:
    species_cols = json.load(f)
with open(model_dir / 'mapping_meta.json') as f:
    meta = json.load(f)
with open(model_dir / 'proxy_map.json') as f:
    proxy_map = {int(k): v for k, v in json.load(f).items()}

NO_LABEL   = int(meta['NO_LABEL'])
n_species  = int(meta['n_species'])
bc_indices = np.load(str(model_dir / 'bc_indices.npy'))

MAPPED_MASK   = bc_indices != NO_LABEL
MAPPED_POS    = np.where(MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_IDX = bc_indices[MAPPED_MASK].astype(np.int32)
print(f'Mapped: {MAPPED_MASK.sum()}/{n_species}  Genus proxy: {len(proxy_map)}')

sample_sub  = pd.read_csv(COMP_DIR / 'sample_submission.csv')
sub_species = [c for c in sample_sub.columns if c != 'row_id']
assert sub_species == species_cols, 'Species column mismatch!'
print(f'Species columns OK: {len(sub_species)}')

Mapped: 203/234  Genus proxy: 4
Species columns OK: 234


In [6]:
# ── Inference helpers ───────────────────────────────────────────────────────
def embed_batch(wavs):
    outs = sess.run(None, {inp_name: wavs})
    return outs[emb_idx].astype(np.float32), outs[logit_idx].astype(np.float32)

def map_logits(logits):
    B   = logits.shape[0]
    out = np.full((B, n_species), logits.mean(axis=1, keepdims=True), dtype=np.float32)
    out[:, MAPPED_POS] = logits[:, MAPPED_BC_IDX]
    for sp_idx, bc_idxs in proxy_map.items():
        out[:, sp_idx] = logits[:, bc_idxs].mean(axis=1)
    return out

def predict_file(fpath, batch_size=8):
    wav_full, _ = librosa.load(str(fpath), sr=SR, mono=True)
    name        = Path(fpath).stem
    segments, end_secs = [], []
    for i, start in enumerate(range(0, len(wav_full), CLIP_SAMPLES)):
        seg = wav_full[start : start + CLIP_SAMPLES]
        if len(seg) < CLIP_SAMPLES:
            seg = np.pad(seg, (0, CLIP_SAMPLES - len(seg)))
        segments.append(seg.astype(np.float32))
        end_secs.append((i + 1) * int(CLIP_SEC))
    rows = []
    for b in range(0, len(segments), batch_size):
        batch       = np.stack(segments[b : b + batch_size])
        emb, lgts   = embed_batch(batch)
        head_probs  = head.predict(emb, verbose=0)
        perch_probs = 1.0 / (1.0 + np.exp(-map_logits(lgts)))
        blended     = perch_weight * perch_probs + (1.0 - perch_weight) * head_probs
        for j, preds in enumerate(blended):
            row = {'row_id': f'{name}_{end_secs[b + j]}'}
            for col, p in zip(species_cols, preds):
                row[col] = float(np.clip(p, 0.0, 1.0))
            rows.append(row)
    return rows

print('Helpers ready.')

Helpers ready.


In [7]:
# ── Run inference (stem-based — same approach as working notebooks) ─────────
sample_sub  = pd.read_csv(COMP_DIR / 'sample_submission.csv')
sub_species = [c for c in sample_sub.columns if c != 'row_id']
sample_sub['stem']    = sample_sub['row_id'].str.rsplit('_', n=1).str[0]
sample_sub['end_sec'] = sample_sub['row_id'].str.rsplit('_', n=1).str[1].astype(int)
unique_stems = sample_sub['stem'].unique()
print(f'Test files: {len(unique_stems)} | rows: {len(sample_sub)}')

test_dir = COMP_DIR / 'test_soundscapes'
all_rows = []

t0 = time.time()
for i, stem in enumerate(unique_stems):
    fpath = test_dir / f'{stem}.ogg'
    if not fpath.is_file():
        print(f'  Missing: {fpath}')
        continue
    y_full, _ = librosa.load(str(fpath), sr=SR, mono=True)
    rows_for_file = sample_sub[sample_sub['stem'] == stem]

    segs, row_ids = [], []
    for ridx, r in rows_for_file.iterrows():
        end_samp   = int(r['end_sec'] * SR)
        start_samp = max(0, end_samp - CLIP_SAMPLES)
        seg = y_full[start_samp:end_samp]
        if len(seg) < CLIP_SAMPLES:
            seg = np.pad(seg, (0, CLIP_SAMPLES - len(seg)))
        segs.append(seg.astype(np.float32))
        row_ids.append(r['row_id'])

    emb, lgts   = embed_batch(np.stack(segs))
    head_probs  = head.predict(emb, verbose=0)
    perch_probs = 1.0 / (1.0 + np.exp(-map_logits(lgts)))
    blended     = perch_weight * perch_probs + (1.0 - perch_weight) * head_probs

    for j, row_id in enumerate(row_ids):
        row = {'row_id': row_id}
        for col, p in zip(species_cols, blended[j]):
            row[col] = float(np.clip(p, 0.0, 1.0))
        all_rows.append(row)

    if (i + 1) % 25 == 0:
        print(f'  [{i+1}/{len(unique_stems)}]  {(i+1)/(time.time()-t0):.1f} files/s')

print(f'Done in {time.time()-t0:.1f}s')
print(f'Prediction rows: {len(all_rows)}')

Test files: 1 | rows: 3
  Missing: /kaggle/input/competitions/birdclef-2026/test_soundscapes/BC2026_Test_0001_S05_20250227_010002.ogg
Done in 0.0s
Prediction rows: 0


In [8]:
# ── Save submission ─────────────────────────────────────────────────────────
print(f'Prediction rows generated: {len(all_rows)}')

if all_rows:
    sub = sample_sub[['row_id']].merge(pd.DataFrame(all_rows), on='row_id', how='left')
    sub[sub_species] = sub[sub_species].fillna(0.0).clip(0.0, 1.0)
else:
    # No test audio available yet (development phase) — submit zeros as placeholder
    print('WARNING: No test soundscapes found. Submitting placeholder zeros.')
    sub = sample_sub.copy()
    sub[sub_species] = 0.0

sub.to_csv('/kaggle/working/submission.csv', index=False)
print(f'submission.csv saved  shape={sub.shape}')
sub.head(3)

Prediction rows generated: 0
submission.csv saved  shape=(3, 237)


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1,stem,end_sec
0,BC2026_Test_0001_S05_20250227_010002_5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BC2026_Test_0001_S05_20250227_010002,5
1,BC2026_Test_0001_S05_20250227_010002_10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BC2026_Test_0001_S05_20250227_010002,10
2,BC2026_Test_0001_S05_20250227_010002_15,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BC2026_Test_0001_S05_20250227_010002,15
